In [28]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
import numpy as np

In [29]:
def create_graph(n=200, p=0.05, features=16, classes=3):
    adj = (torch.rand(n, n) < p).float()
    adj = torch.triu(adj, 1)
    adj = adj + adj.t() # Undirected

    x = torch.randn(n, features)
    y = torch.randint(0, classes, (n,))

    train_mask = torch.zeros(n, dtype=torch.bool)
    test_mask = torch.zeros(n, dtype=torch.bool)

    train_mask[:int(0.7*n)] = True
    test_mask[:int(0.7*n)] = True

    return adj, x, y, train_mask, test_mask

In [30]:
create_graph()

(tensor([[0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         ...,
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.],
         [0., 0., 0.,  ..., 0., 0., 0.]]),
 tensor([[-0.3961,  0.8204,  0.5396,  ..., -0.5410, -1.4911,  1.0338],
         [-0.9173, -0.2647,  0.4604,  ...,  0.6184, -1.3008, -0.5645],
         [ 0.0362, -0.1325, -3.0955,  ..., -0.5390, -3.5917, -0.4368],
         ...,
         [-1.5070, -1.1623,  0.6411,  ..., -1.4674, -0.8186, -0.3044],
         [ 1.1905, -1.0995, -0.2139,  ..., -0.8072,  0.7985, -0.9285],
         [ 2.2695, -0.2721,  0.3771,  ..., -0.6878,  0.8682, -0.5522]]),
 tensor([2, 1, 1, 1, 0, 2, 2, 0, 2, 0, 1, 0, 2, 0, 2, 1, 0, 2, 2, 1, 2, 0, 2, 1,
         2, 2, 1, 0, 0, 2, 2, 2, 2, 1, 2, 2, 0, 1, 0, 1, 0, 2, 2, 2, 0, 2, 1, 1,
         1, 2, 0, 0, 1, 1, 1, 2, 2, 2, 2, 0, 1, 2, 2, 1, 0, 1, 0, 1, 2, 2, 0, 2,
         1, 1, 1, 0, 0, 0, 2, 2, 2, 2, 1, 0, 2, 2, 1,

In [31]:
def normalize_adj(A):
    I = torch.eye(A.size(0))
    A_hat = A + I
    D = torch.diag(torch.sum(A_hat, dim=1))
    D_inv_sqrt = torch.inverse(torch.sqrt(D))
    return D_inv_sqrt @ A_hat @ D_inv_sqrt

In [32]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.W1 = nn.Parameter(torch.randn(in_dim, hidden))
        self.W2 = nn.Parameter(torch.randn(hidden, out_dim))

    def forward(self, A_norm, X):
        H = torch.relu(A_norm @ X @ self.W1) # H = n*d^ size [ n*n n*d d*d^ = n*d^]
        H = A_norm @ H @ self.W2
        return H

In [61]:
class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, num_samples=5):
        super().__init__()
        self.W1 = nn.Linear(in_dim * 2, hidden)
        self.W2 = nn.Linear(hidden * 2, out_dim)
        self.num_samples = num_samples
    
    def sample_neighbors(self, A, nodes):
        sampled_neigbors = []

        for v in nodes:
            neigh = (A[v] > 0).nonzero().squeeze()

            if neigh.numel() == 0:
                sampled_neigbors.append(torch.tensor([], dtype=torch.long))
                continue

            perm = torch.randperm(neigh.numel())
            neigh = neigh[perm[:self.num_samples]]
            sampled_neigbors.append(neigh)

        return sampled_neigbors


    def aggregate(self, X, sampled_neighbors):
        out = torch.zeros_like(X)

        for i, neigh in enumerate(sampled_neighbors):
            if len(neigh) == 0:
                out[i] = 0
            else:
                out[i] = X[neigh].mean(dim=0)
        
        return out
    
    def forward(self, A, X):
        N = X.size(0)
        nodes = torch.arange(N)

        # Layer 1
        neigh1 = self.sample_neighbors(A, nodes)
        agg1 = self.aggregate(X, neigh1)

        H1 = torch.relu(self.W1(torch.cat([X, agg1], dim=1)))

        # Layer 2
        neigh2 = self.sample_neighbors(A, nodes)
        agg2 = self.aggregate(H1, neigh2)

        H2 = torch.relu(self.W2(torch.cat([H1, agg2], dim=1)))

        return H2

In [62]:
class GAT(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, hidden, bias=False)
        self.a = nn.Parameter(torch.randn(2 * hidden, 1))
        self.W_out = nn.Linear(hidden, out_dim)

    def forward(self, A, X):
        H = self.W(X)
        N = H.size(0)

        e = torch.zeros(N, N)

        for i in range(N):
            for j in range(N):
                if A[i, j] > 0:
                    e[i, j] = torch.relu(
                        torch.cat([H[i], H[j]]) @ self.a
                    )

        alpha = torch.softmax(e, dim=1)

        H_new = alpha @ H

        return self.W_out(H_new)

In [63]:
def train(model, A, X, y, train_mask, epochs=200):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    if isinstance(model, GCN):
        A_input = normalize_adj(A)
    else:
        A_input = A

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        out = model(A_input, X)
        loss = F.cross_entropy(out[train_mask], y[train_mask])

        loss.backward()
        optimizer.step()

        if epoch % 50 == 0:
            print(f"Epoch {epoch}, Loss: {loss.item():.4f}")
        
    return model

In [64]:
def test(model, A, X, y, mask):
    model.eval()

    if isinstance(model, GCN):
        A_input = normalize_adj(A)
    else:
        A_input = A


    out = model(A_input, X)
    pred = out.argmax(dim=1)

    acc = (pred[mask] == y[mask]).float().mean()

    return acc.item()

In [65]:
A, X, y, train_mask, test_mask  = create_graph()

models = {
    "GCN": GCN(X.shape[1], 32, len(torch.unique(y))),
    "GraphSAGE": GraphSAGE(X.shape[1], 32, len(torch.unique(y))),
    "GAT": GAT(X.shape[1], 16, len(torch.unique(y)))
}

In [66]:
models

{'GCN': GCN(),
 'GraphSAGE': GraphSAGE(
   (W1): Linear(in_features=32, out_features=32, bias=True)
   (W2): Linear(in_features=64, out_features=3, bias=True)
 ),
 'GAT': GAT(
   (W): Linear(in_features=16, out_features=16, bias=False)
   (W_out): Linear(in_features=16, out_features=3, bias=True)
 )}

In [67]:
for name, model in models.items():
    print(f"Training {name}")
    model = train(model, A, X, y, train_mask)

    acc = test(model, A, X, y, test_mask)

    print(f"{name} Test Accuracy: {acc:.4f}")

Training GCN
Epoch 0, Loss: 2.2607
Epoch 50, Loss: 0.9300
Epoch 100, Loss: 0.7969
Epoch 150, Loss: 0.6841
GCN Test Accuracy: 0.8286
Training GraphSAGE
Epoch 0, Loss: 1.1083
Epoch 50, Loss: 0.1818
Epoch 100, Loss: 0.0182
Epoch 150, Loss: 0.0115
GraphSAGE Test Accuracy: 1.0000
Training GAT
Epoch 0, Loss: 1.0940
Epoch 50, Loss: 0.8023
Epoch 100, Loss: 0.6889
Epoch 150, Loss: 0.6165
GAT Test Accuracy: 0.6857
